# Phenotype Generation: Longest Therapy Duration for Each Drug


**Objective:**

The phenotype is defined as the **duration of the longest continuous therapy** for a given drug. This analysis is focused on drugs from a predefined list, which is available [here](https://docs.google.com/spreadsheets/d/1eyPvwoZVXpSKPt9yt0N-jycmwcpYd-F0ySRb9OtJ-ik/edit?gid=0#gid=0). For this analysis, a therapy is considered continuous unless there is a gap between prescriptions exceeding 60 days plus the number of pills from the preceding prescription. This adjusted calculation better accounts for a patient's usage patterns. Additionally, for patients with only a single prescription for a given drug, we assume the number of pills dispensed is equal to the duration of the therapy. A phenotype value will only be assigned to individuals who have at least one prescription for this specific drug. Patients without any record of taking the drug will have a null value for this phenotype. Addictionaly for each of duration of the longest continuous therapy phenotype transformed phenotype is created using the natural logarithm. 

**Output:**

Phenotype name: `<drug_name>__longest_therapy_duration`

Outliers: For each drug phenotype, the standard deviation (σ) is calculated independently. Maximum value for each phenotype is set as μ+8σ, any phenotype value will be capped to this value.

Transformed phenotype name: `<drug_name>__longest_therapy_duration__ln` 



In [ ]:
import pyspark
import dxpy
import hail as hl
import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import json
from scipy.stats import boxcox

In [ ]:
sc = pyspark.SparkContext()
spark = pyspark.sql.SparkSession(sc)
hl.init(sc=sc, default_reference='GRCh38')

In [ ]:
from datetime import datetime
print(f'Timestamp: {datetime.now()}')
print(f'Instance type: {dxpy.describe(dxpy.JOB_ID)["instanceType"]}')
print(f'Hail version: {hl.version()}')
print(f'Spark version: {spark.version}')

### Configuration and Hail tables loading

In [ ]:
input_database = 'prescriptions_db'
input_prescriptions_tb = 'cleaned_prescriptions_with_doses_v6.2.0.ht'

tmp_output_database = 'prescriptions_db'
tmp_output_tb = 'not_cleaned_longest_therapy_duaration_v6_2_0.ht'

output_database = 'prescriptions_db'
output_tb = 'longest_therapy_duration_phenotypes_v6_2_0.ht'

In [ ]:
input_db_id = dxpy.find_one_data_object(name=input_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
input_prescriptions_ht = hl.read_table(f'dnax://{input_db_id}/{input_prescriptions_tb}')

In [ ]:
input_prescriptions_ht = input_prescriptions_ht.drop(
    'drug_name',
    'tokenized_drug_name',
    'matched_code',
    'match_mode'
)

In [ ]:
input_prescriptions_ht.describe()

In [ ]:
start_num_unique_eids = input_prescriptions_ht.aggregate(hl.agg.collect_as_set(input_prescriptions_ht.eid))
start_num_unique_eids = len(start_num_unique_eids)

### Load drug list and filter prescription table

In [ ]:
with open('../../../data/input/substances.json', 'r') as f:
    data = json.load(f)

drug_list = []

for drug_name, _ in data.items():
    drug_list.append(drug_name)
    
drug_list = list(drug_list)

drugs_to_filter = hl.literal(drug_list)

In [ ]:
ht = input_prescriptions_ht.filter(
    drugs_to_filter.contains(input_prescriptions_ht.substance)
)
ht = ht.persist()

In [ ]:
ht = ht.annotate(
    date_struct = hl.struct(
        year = hl.int32(ht.date.split('-')[0]),
        month = hl.int32(ht.date.split('-')[1]),
        day = hl.int32(ht.date.split('-')[2])
    )
)
ht = ht.drop('date')
ht = ht.persist()

### Add intervals and calculate therapies durations

In [ ]:
ht_filtered = ht.group_by(ht.eid, ht.substance).aggregate(
    all_data_collected = hl.agg.collect(
        hl.struct(
            date_struct = ht.date_struct,
            quantity = ht.quantity,
            doses = ht.doses
        )
    )
)

ht_filtered = ht_filtered.annotate(
    all_data_collected_sorted = hl.sorted(
        ht_filtered.all_data_collected,
        key=lambda s: (s.date_struct.year, s.date_struct.month, s.date_struct.day)
    )
).drop('all_data_collected')

ht_final = ht_filtered.annotate(
    data_with_next = hl.range(0, hl.len(ht_filtered.all_data_collected_sorted)).map(lambda i:
        hl.struct(
            date_struct = ht_filtered.all_data_collected_sorted[i].date_struct,
            quantity = ht_filtered.all_data_collected_sorted[i].quantity,
            doses = ht_filtered.all_data_collected_sorted[i].doses,
            next_date_struct = hl.if_else(
                i < hl.len(ht_filtered.all_data_collected_sorted) - 1,
                ht_filtered.all_data_collected_sorted[i + 1].date_struct,
                hl.missing(ht_filtered.all_data_collected_sorted.date_struct.dtype.element_type)
            ),
            next_quantity = hl.if_else(
                i < hl.len(ht_filtered.all_data_collected_sorted) - 1,
                ht_filtered.all_data_collected_sorted[i + 1].quantity,
                hl.missing(ht_filtered.all_data_collected_sorted.quantity.dtype.element_type)
            ),
            next_doses = hl.if_else(
                i < hl.len(ht_filtered.all_data_collected_sorted) - 1,
                ht_filtered.all_data_collected_sorted[i + 1].doses,
                hl.missing(ht_filtered.all_data_collected_sorted.doses.dtype.element_type)
            )
        )
    )
).explode('data_with_next')

ht_final = ht_final.annotate(
    date_struct = ht_final.data_with_next.date_struct,
    quantity = ht_final.data_with_next.quantity,
    doses = ht_final.data_with_next.doses,
    next_date_struct = ht_final.data_with_next.next_date_struct,
    next_quantity = ht_final.data_with_next.next_quantity,
    next_doses = ht_final.data_with_next.next_doses
).drop('data_with_next', 'all_data_collected_sorted')

In [ ]:
ht_final = ht_final.persist()

In [ ]:
def days_since_epoch(y, m, d):
    days_in_month_array = hl.array([31, 28, 31, 30, 31, 30, 31, 31, 30, 31, 30, 31])
    is_leap = (y % 4 == 0) & (y % 100 != 0) | (y % 400 == 0)
    days_in_feb = hl.if_else(is_leap, 29, 28)
    days_from_months = hl.sum(hl.range(0, m - 1).map(lambda i:
        hl.if_else(i == 1, days_in_feb, days_in_month_array[i])
    ))
    days_from_year = hl.sum(hl.range(1940, y).map(lambda year: hl.if_else((year % 4 == 0) & (year % 100 != 0) | (year % 400 == 0), 366, 365)))
    return days_from_year + days_from_months + d

In [ ]:
ht_final = ht_final.annotate(
    interval = hl.if_else(
        hl.is_missing(ht_final.next_date_struct),
        hl.missing(hl.tint32),
        days_since_epoch(
            ht_final.next_date_struct.year,
            ht_final.next_date_struct.month,
            ht_final.next_date_struct.day
        ) - days_since_epoch(
            ht_final.date_struct.year,
            ht_final.date_struct.month,
            ht_final.date_struct.day
        )
    )
).drop('next_date_struct', 'next_quantity', 'next_doses')

In [ ]:
ht_final = ht_final.annotate(
    interval = hl.if_else(
        hl.is_missing(ht_final.interval),
        ht_final.quantity.value,
        ht_final.interval
    )
)

In [ ]:
ht_final = ht_final.persist()

In [ ]:
ht = ht_final.order_by(ht_final.eid, ht_final.substance, ht_final.date_struct.year, ht_final.date_struct.month, ht_final.date_struct.day)
ht = ht.persist()

In [ ]:
ht_with_blocks = ht.annotate(
    is_new_block = (ht.interval - ht.quantity.value) > 60
)
ht_with_blocks = ht_with_blocks.annotate(
    interval = hl.if_else(
        ht_with_blocks.is_new_block,
        ht_with_blocks.quantity.value,
        ht_with_blocks.interval
    )
)

In [ ]:
ht_re_grouped = ht_with_blocks.group_by(ht_with_blocks.eid, ht_with_blocks.substance).aggregate(
    data = hl.agg.collect(
        hl.struct(
            date_struct = ht_with_blocks.date_struct,
            quantity = ht_with_blocks.quantity,
            doses = ht_with_blocks.doses,
            interval = ht_with_blocks.interval,
            is_new_block = ht_with_blocks.is_new_block
        )
    )
)
ht_re_grouped = ht_re_grouped.annotate(
    data = hl.sorted(
        ht_re_grouped.data,
        key=lambda s: (s.date_struct.year, s.date_struct.month, s.date_struct.day)
    )
)

In [ ]:
ht_with_block_id = ht_re_grouped.annotate(
    data = ht_re_grouped.data.scan(
        lambda i, r: hl.struct(
            date_struct=r.date_struct,
            quantity=r.quantity,
            doses=r.doses,
            interval=r.interval,
            is_new_block=r.is_new_block,
            block_id=hl.if_else(r.is_new_block, i.block_id + 1, i.block_id)
        ),
        hl.struct(
            date_struct=hl.struct(year=hl.int32(0), month=hl.int32(0), day=hl.int32(0)),
            quantity=hl.struct(is_days=hl.bool(False), value=hl.int64(0)),
            doses=hl.struct(value=hl.int64(0), unit=hl.str("")),
            interval=hl.int64(0),
            is_new_block=hl.bool(False),
            block_id=hl.int32(0)
        )
    )
)
ht_with_block_id = ht_with_block_id.persist()

In [ ]:
ht_flat = ht_with_block_id.explode('data')

ht_flat = ht_flat.annotate(
    date_struct = ht_flat.data.date_struct,
    quantity = ht_flat.data.quantity,
    doses = ht_flat.data.doses,
    interval = ht_flat.data.interval,
    is_new_block = ht_flat.data.is_new_block,
    block_id = ht_flat.data.block_id
).drop('data')

In [ ]:
therapy_blocks = ht_flat.group_by(
    ht_flat.eid,
    ht_flat.substance,
    ht_flat.block_id
).aggregate(
    prescriptions=hl.agg.collect(
        hl.struct(
            date_struct=ht_flat.date_struct,
            quantity=ht_flat.quantity,
            doses=ht_flat.doses,
            interval=ht_flat.interval
        )
    ),
    therapy_prescriptions_number=hl.agg.count(),
    therapy_interval_days=hl.agg.sum(ht_flat.interval)
)
therapy_blocks = therapy_blocks.persist()

In [ ]:
longest_therapies = therapy_blocks.group_by(
    therapy_blocks.eid, therapy_blocks.substance
).aggregate(
    max_therapy_interval_days=hl.agg.max(therapy_blocks.therapy_interval_days),
    max_therapy_prescriptions_number=hl.agg.max(therapy_blocks.therapy_prescriptions_number)
).select_globals()

In [ ]:
substance_stats = longest_therapies.group_by(
    longest_therapies.substance
).aggregate(
    interval_stats = hl.agg.stats(longest_therapies.max_therapy_interval_days)
)

In [ ]:
substance_thresholds = substance_stats.annotate(
    mu_plus_8sigma = substance_stats.interval_stats.mean + (
        8 * substance_stats.interval_stats.stdev
    )
)
substance_thresholds = substance_thresholds.persist()

In [ ]:
longest_therapies_with_threshold = longest_therapies.annotate(
    mu_plus_8sigma_limit = substance_thresholds[longest_therapies.substance].mu_plus_8sigma
)

In [ ]:
ht = longest_therapies_with_threshold.annotate(
    capped_duraion = hl.min(
        longest_therapies_with_threshold.max_therapy_interval_days,
        longest_therapies_with_threshold.mu_plus_8sigma_limit
    )
)

In [ ]:
ht = ht.persist()

### How many values was capped?

In [ ]:
total_capped_count = ht.aggregate(
    hl.agg.count_where(
        ht.max_therapy_interval_days > ht.mu_plus_8sigma_limit
    )
)

In [ ]:
total_capped_count

### Final table preparation

In [ ]:
ht = ht.annotate(
    ln_capped_duraion = hl.log(ht.capped_duraion)
)
ht = ht.persist()

In [ ]:
all_substances = ht.aggregate(hl.agg.collect_as_set(ht.substance))

In [ ]:
ht_pivoted_ln = ht.group_by(ht.eid).aggregate(
    **{f'{sub}__longest_therapy_duration__ln': hl.agg.filter(
        ht.substance == sub, 
        hl.agg.max(ht.ln_capped_duraion)
    ) for sub in all_substances}
)

ht_pivoted_ln = ht_pivoted_ln.persist()

In [ ]:
ht_pivoted_ln.count()

In [ ]:
ht_pivoted = ht.group_by(ht.eid).aggregate(
    **{f'{sub}__longest_therapy_duration': hl.agg.filter(
        ht.substance == sub, 
        hl.agg.max(ht.capped_duraion)
    ) for sub in all_substances}
)

ht_pivoted = ht_pivoted.persist()

In [ ]:
ht_pivoted.count()

In [ ]:
ht_pivoted_ln = ht_pivoted_ln.key_by('eid') 
ht_pivoted = ht_pivoted.key_by('eid')

In [ ]:
merged_ht = ht_pivoted.join(
    ht_pivoted_ln, 
    how='outer'
)

In [ ]:
merged_ht = merged_ht.key_by('eid')

In [ ]:
spark.sql(f"CREATE DATABASE IF NOT EXISTS {output_database} LOCATION 'dnax://'")
output_db_id = dxpy.find_one_data_object(name=output_database, classname='database', project=dxpy.PROJECT_CONTEXT_ID)['id']
output_filtered_prescriptions_url = f'dnax://{output_db_id}/{output_tb}'

%time merged_ht.write(output_filtered_prescriptions_url, overwrite=True)